In [0]:
%run "../00_config"

In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import functions as F

##Fetch weather for all 6 saudi cities

In [0]:
def fetch_weather(city: dict) -> dict:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude":        city["lat"],
        "longitude":       city["lon"],
        "daily":           ["temperature_2m_max",
                            "temperature_2m_min",
                            "apparent_temperature_max",
                            "precipitation_sum",
                            "windspeed_10m_max",
                            "winddirection_10m_dominant"],
        "timezone":        "Asia/Riyadh",
        "forecast_days":   1
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        data   = r.json()
        daily  = data.get("daily", {})
        return {
            "city":                     city["city"],
            "latitude":                 str(city["lat"]),
            "longitude":                str(city["lon"]),
            "date":                     str(daily.get("time",                       [None])[0]),
            "temperature_max":          str(daily.get("temperature_2m_max",         [None])[0]),
            "temperature_min":          str(daily.get("temperature_2m_min",         [None])[0]),
            "apparent_temperature_max": str(daily.get("apparent_temperature_max",   [None])[0]),
            "precipitation_sum":        str(daily.get("precipitation_sum",          [None])[0]),
            "windspeed_max":            str(daily.get("windspeed_10m_max",          [None])[0]),
            "wind_direction_dominant":  str(daily.get("winddirection_10m_dominant", [None])[0]),
            "_snapshot_date":           SNAPSHOT_DATE,
            "_ingested_at":             datetime.now(timezone.utc).isoformat()
        }
    except Exception as e:
        print(f"  Error for {city['city']}: {e}")
        return {}

weather_records = []
for city in SAUDI_CITIES:
    record = fetch_weather(city)
    if record:
        weather_records.append(record)
        print(f"✓ {city['city']} → temp_max: {record['temperature_max']}°C  wind: {record['windspeed_max']} km/h")

print(f"\nTotal records: {len(weather_records)}")

#Write to Bronze

## Deduplicate and Keep one record per city per date

In [0]:
df_new = spark.createDataFrame(weather_records)

# Normalize city name
df_new = df_new.withColumn("city", F.initcap(F.col("city")))

# Deduplicate within new batch
df_new = df_new.dropDuplicates(["city", "date"])

# Get existing city+date combinations
existing_keys = spark.table(BRONZE_WEATHER) \
    .select("city", "date")

# Only keep new records that don't exist yet
df_to_append = df_new.join(
    existing_keys,
    on=["city", "date"],
    how="left_anti"
)

if df_to_append.count() > 0:
    (df_to_append
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_WEATHER)
    )
    print(f"✅ {df_to_append.count()} weather records written to {BRONZE_WEATHER}")
else:
    print("⚠️ No new records — today's data already exists in Bronze")
display(df_to_append)

##Validate

In [0]:
spark.table(BRONZE_WEATHER) \
    .select("city", "date", "temperature_max", "temperature_min",
            "precipitation_sum", "windspeed_max") \
    .show(truncate=False)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.bronze.weather;